In [1]:
import pandas as pd
from pathlib import Path

Bronze_file = Path("bronze_layer")/"bravo_13axsam_bronze_2026-02-13_2026-02-13T19-49-32Z.csv"

df=pd.read_csv(Bronze_file,encoding="utf-8-sig")

print(len(df))

df.head(1)

468


,scrape_run_id,scrape_ts_utc,snapshot_date,market,category_raw,product_name_raw,barcode,price_current_azn,price_old_azn,price_current_raw,price_old_raw,promo_flag,purchasable_balance,availability_status,unit_info_raw,product_hash,dq_missing_barcode,dq_invalid_price,dq_missing_purchasable_balance
0,878adaa7-b4f8-4c2b-907d-8fedd621d9e7,2026-02-13T19:49:32.419525,2026-02-13,Bravo,ayran,Atena Ayran Nanəli 1.4% 1l,4.760088e+12,1.99,2.49,199,249,1,6.0,available,1 əd.,f87d5b94de5d4d2bd0782e46f7924df72ebebf82609cda...,0,0,0


In [2]:
df["scrape_ts_utc"].dtype
df["snapshot_date"].dtype
df["scrape_ts_utc"]=pd.to_datetime(df["scrape_ts_utc"],errors="coerce")
df["snapshot_date"]=pd.to_datetime(df["snapshot_date"],errors="coerce").dt.date
df["scrape_ts"]=pd.to_datetime(df["scrape_ts_utc"],errors="coerce",utc=True)
df["scrape_ts_baku"]=df["scrape_ts"].dt.tz_convert("Asia/Baku")
df["snapshot_date"] = pd.to_datetime(df["scrape_ts"]).dt.date

df["baku_hour"]=df["scrape_ts_baku"].dt.hour

df["session"]=None

df.loc[df["baku_hour"].between(6,11),"session"]="morning"
df.loc[df["baku_hour"].between(18,23),"session"]="evening"

df["market_day"]=df["market"].astype(str)+"|"+df["snapshot_date"].astype(str)
df["market_day_session"] = df["market_day"].astype(str) + "|" + df["session"].astype(str)

In [ ]:

import re
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
UNIT_LIST = r"kg|kq|qram|qr|gr|g|litr|lt|l|ml"

_MP_FORWARD = re.compile(
    rf"(\d+)\s*[x×\*]\s*(\d+(?:[.,]\d+)?)\s*({UNIT_LIST})\b",
    re.I
)

_MP_REVERSE = re.compile(
    rf"(\d+(?:[.,]\d+)?)\s*({UNIT_LIST})\s*[x×\*]\s*(\d+)\b",
    re.I
)


_SINGLE = re.compile(
    rf"(\d+(?:[.,]\d+)?)\s*({UNIT_LIST})\b",
    re.I
)


_THOU_RE = re.compile(r"(\d)[.,](\d{3})(?!\d)")


def _parse(s: str) -> float:
    """'1.820' → 1820 · '1,8' → 1.8 · '1.4' → 1.4"""
    s = s.strip()
    if re.match(r"^\d+[.,]\d{3}$", s):          
        return float(s.replace(".", "").replace(",", ""))
    return float(s.replace(",", "."))


def _to_base(qty: float, unit: str) -> tuple[float, str]:
    """Hər şeyi ya g ya ml-ə çevir."""
    u = unit.lower()
    if u in ("kg", "kq"):                return qty * 1000, "g"
    if u in ("g", "gr", "qr", "qram"):  return qty,        "g"
    if u in ("l", "lt", "litr"):         return qty * 1000, "ml"
    if u == "ml":                         return qty,        "ml"
    return np.nan, None


def extract_std_qty(text: object) -> tuple:
    """
    Məhsul adından kəmiyyət + vahid çıxarır.

    Qaydalar (prioritet sırası):
      A) MultiPack forward : 6x200ml / 6 × 500 g
      B) MultiPack reverse : 500g x 6
      C) Single pack       : 200 ml, 1.5l, 800qr, 1.820qr
      D) Tapılmadısa       : (NaN, None, 1)
    """
    if not isinstance(text, str) or not text.strip():
        return np.nan, None, 1

    
    t = text.lower().replace("\xa0", " ").replace(",", ".")
    
    t = _THOU_RE.sub(r"\1\2", t)

  
    m = _MP_FORWARD.search(t)
    if m:
        count = int(m.group(1))
        qty   = _parse(m.group(2))
        base_qty, base_unit = _to_base(qty * count, m.group(3))
        return base_qty, base_unit, count

    m = _MP_REVERSE.search(t)
    if m:
        qty   = _parse(m.group(1))
        count = int(m.group(3))
        base_qty, base_unit = _to_base(qty * count, m.group(2))
        return base_qty, base_unit, count

    all_m = _SINGLE.findall(t)
    if all_m:
        qty_str, unit = all_m[-1]
        base_qty, base_unit = _to_base(_parse(qty_str), unit)
        return base_qty, base_unit, 1


    return np.nan, None, 1

df[["std_qty_total", "std_unit", "pack_multiplier"]] = (
    df["product_name_raw"]
    .apply(lambda x: pd.Series(extract_std_qty(x)))
)


df["dq_missing_unit"] = (
    df["std_qty_total"].isna() | (df["std_qty_total"] <= 0)
).astype(int)


if "price_current_azn" not in df.columns:
    df["price_current_azn"] = np.nan

_ok = (df["dq_missing_unit"] == 0) & df["price_current_azn"].notna()

df["unit_price_1kg"] = np.where(
    _ok & (df["std_unit"] == "g"),
    df["price_current_azn"] / (df["std_qty_total"] / 1000),
    np.nan
)

df["unit_price_1l"] = np.where(
    _ok & (df["std_unit"] == "ml"),
    df["price_current_azn"] / (df["std_qty_total"] / 1000),
    np.nan
)

df["unit_price_base"] = np.where(
    df["std_unit"] == "g",   df["unit_price_1kg"],
    np.where(df["std_unit"] == "ml", df["unit_price_1l"], np.nan)
)


_labels = {
    "g":  "AZN/kg",
    "ml": "AZN/L",
}
df["unit_price_base_label"] = df["std_unit"].map(_labels)


df["unit_price_raw"] = np.where(
    _ok, df["price_current_azn"] / df["std_qty_total"], np.nan
)


df["name_lc"] = df["product_name_raw"].str.lower().str.strip()




COLS = [
    "product_name_raw",
    "name_lc",
    "std_qty_total",
    "std_unit",
    "pack_multiplier",
    "dq_missing_unit",
    "unit_price_1kg",
    "unit_price_1l",
    "unit_price_base",
    "unit_price_base_label",
    "unit_price_raw",
]
df_out = df[COLS]




total   = len(df_out)
ok_cnt  = (df_out["dq_missing_unit"] == 0).sum()
fail_cnt = df_out["dq_missing_unit"].sum()
g_cnt   = (df_out["std_unit"] == "g").sum()
ml_cnt  = (df_out["std_unit"] == "ml").sum()
mp_cnt  = (df_out["pack_multiplier"] > 1).sum()

print("\n" + "="*50)
print("  UNIT EXTRACTION REPORT")
print("="*50)
print(f"  Cemi mehsul          : {total}")
print(f"  Ugurlu parse         : {ok_cnt}  ({ok_cnt/total*100:.1f}%)")
print(f"  Tapilmayan (DQ flag) : {fail_cnt}")
print(f"  Catkin olan (g)      : {g_cnt}")
print(f"  Mayelik olan (ml)    : {ml_cnt}")
print(f"  Multipack            : {mp_cnt}")
print("="*50)


fail_df = df_out.loc[df_out["dq_missing_unit"] == 1, "product_name_raw"]
print(f"\nUnit tapilmayan {fail_cnt} mehsul:")
print(fail_df.to_string(index=False))



  UNIT EXTRACTION REPORT
  Cemi mehsul          : 468
  Ugurlu parse         : 451  (96.4%)
  Tapilmayan (DQ flag) : 17
  Catkin olan (g)      : 280
  Mayelik olan (ml)    : 171
  Multipack            : 4

Unit tapilmayan 17 mehsul:
                      DR.Milk Ayran
              Düyü Tamasa Miad 910q
        Tamasa Miad Düyü 1.820q Pet
          Doyaruşka Kərə Yağı 380 q
        Dr.Oetker Qabartma Tozu 5əd
                Dr.Oetker Vanil 5əd
                 Atena Uht Süd 2.4%
Vitaminli Pirinc Unu 12x200 Soyiğit
       Milvana Kənd Yumurtası 10əd.
      Milvana Qəhvəyi Yumurta 10əd.
      Milvana Qəhvəyi Yumurta 15əd.
           Milvana Ağ Yumurta 10əd.
           Milvana Ağ Yumurta 15əd.
 Hacıqabul Yumurta İri Ağ 1əd 8356
  Hacıqabul Kənd Yumurtası 1əd 7262
           Yumurta N1 Kənd (10 əd.)
   Vanish Ləkə Təmizləyici Toz 500q


In [4]:
df = df[~df.astype(str).apply(lambda x: x.str.contains('Qatiq 700q', case=False)).any(axis=1)]
df = df[~df.astype(str).apply(lambda x: x.str.contains('Kefir', case=False)).any(axis=1)]
df = df[~df.astype(str).apply(lambda x: x.str.contains('Cream', case=False)).any(axis=1)]
df = df[~df.astype(str).apply(lambda x: x.str.contains('Vanil', case=False)).any(axis=1)]
df = df[~df.astype(str).apply(lambda x: x.str.contains('Soyiğit', case=False)).any(axis=1)]
df = df[~df.astype(str).apply(lambda x: x.str.contains('yumurtası', case=False)).any(axis=1)]
df = df[~df.astype(str).apply(lambda x: x.str.contains('Yumurtası', case=False)).any(axis=1)]
df = df[~df.astype(str).apply(lambda x: x.str.contains('Yumurta', case=False)).any(axis=1)]


In [6]:
indicates_to_drop=[27,267,310,430]
df=df.drop(indicates_to_drop)

In [7]:
df[df["std_qty_total"].isna()][
    ["product_name_raw", "category_raw"]
].head(20)

,product_name_raw,category_raw
61,Düyü Tamasa Miad 910q,duyu
62,Tamasa Miad Düyü 1.820q Pet,duyu
116,Doyaruşka Kərə Yağı 380 q,kereyagi


In [ ]:

import re
import pandas as pd

_AZ = str.maketrans({
    "ə":"e","Ə":"e","ş":"s","Ş":"s","ı":"i","İ":"i",
    "ğ":"g","Ğ":"g","ç":"c","Ç":"c","ö":"o","Ö":"o","ü":"u","Ü":"u",
})

def fold(x: object) -> str:
    if not isinstance(x, str): return ""
    return re.sub(r"\s+", " ", x.lower().translate(_AZ).strip())


BRANDS = [
   
    "Dr. Milk", "DR.Milk", "Mr Muscle", "Mr. Muscle", "Mr Pepper",
    "Zolotoe Solnışko", "Zolotoe Solnishko",
    "Port De Kalleh",
    "Zeytun Bağları", "Zeytun Baglari",
    "Ana Bala",
    "Seçmə Şəkər",
    "Azər Şəkər", "Azərşəkər", "AzerSüd", "Azərsüd",
    "Krestyanskoe",
    "Prostokvaşino", "Prostokvashino",
    "Slavyanskoe", "Slavyanskoye",
    "Belovejskiye",
    "Belaya Korova", "Belaya",
    "French Cow", "French Garden",
    "Best Cow",
    "Green Fields", "Green Milk",
    "Golden Fields", "Golden Sun",
    "Best Food",
    "Pasta Reggia",
    "Dr. Oetker", "Dr.Oetker",
    "N1",
    "ABC", "Absheron", "Active", "Age", "Alpro", "Anadolu", "Anchor",
    "Ankara",
    "Aquşa",
    "Ariel",
    "Ashrafi", "Astara", "Atena",
    "Azərduz",
    "Barilla", "BASSO", "Basso", "Belmas",
    "Besta", "Bingo", "Bismak", "Bizim", "Biznes",
    "Borges", "Brest",
    "Cam", "Cleverest", "Consul", "Costa", "Crown",
    "Danone", "Der", "Divo", "Domik", "Doymak", "Doyaruşka",
    "Era", "Evim", "Eximo", "Extra",
    "Favelli", "Femilea", "Final",
    "Gamarjobo", "Garnec", "Gerda",
    "Harika", "Himalay", "Homogenize",
    "Kalinka", "Kalleh", "Kalyon", "Kerem",
    "Kikkoman", "Komili",
    "Kırlangıç", "Kırlanğıc", "Kirlangic",
    "Lubino", "Luck", "Luglio",
    "Mainland", "Makfa", "Maqiya",
    "Marjan", "Miad", "Milkidy", "Milla",
    "Mistral", "Mohab", "Monini",
    "Mutlu", "Möcüzə", "Mocuze",
    "Naş", "Novaya",
    "Oil", "Olitalia", "Olivea", "Oluna", "Oman", "Ozmaya",
    "Pakmaya", "Peros", "Persil", "Perwoll",
    "President", "Pudov", "Pınar",
    "Qidam", "Qranit",
    "Richmond", "Rollton",
    "Sab", "Saf", "Salute", "Sana", "Saville", "Savuşkin",
    "Schar", "Sedraya", "Shaan",
    "Sizlik", "Slavyan", "Sloboda",
    "Sultan", "Sun", "Sunar", "Sunbai", "Svalya",
    "Synergetic",
    "Tala", "Tamasa", "Tamasha", "Tamaşa", "Tanelli",
    "Tibet", "Tide", "Tsar",
    "Ush", "Uzun",
    "Valio", "Vanish", "Vernel", "Vitadiet", "Vugma",
    "Yonca", "Yurdum",
    "Zade", "Zeytuna", "Zira", "Zolotoe", "Zoohra",
    "İvanovka", "İçim",
    "Şamtaz", "Şifay",
]


fold_to_canon: dict[str, str] = {}
for b in BRANDS:
    fold_to_canon[fold(b)] = b

_sorted_folds = sorted(fold_to_canon.keys(), key=len, reverse=True)


_BRAND_PAT = re.compile(
    r"(?:(?:^|(?<=[^a-z0-9]))(PLACEHOLDER)(?:(?=$)|(?=[^a-z0-9])))",
    re.I
)
_BRAND_PAT = re.compile(
    r"(?:(?<=^)|(?<=[^a-z0-9]))("
    + "|".join(re.escape(b) for b in _sorted_folds)
    + r")(?:(?=$)|(?=[^a-z0-9]))",
    re.I,
)


_ALIASES: list[tuple[re.Pattern, str]] = [
    (re.compile(r"^d[üu]y[üu]\s*unu\b",       re.I), "Astara Unu"),   
    (re.compile(r"^qatiq\b",                   re.I), "İvanovka"),     
    (re.compile(r"^qaymaq\b",                  re.I), "Qaymaq"),          (re.compile(r"\bdr\.?\s*milk\b",          re.I), "Dr. Milk"),
    (re.compile(r"\bmr\.?\s*muscle\b",        re.I), "Mr Muscle"),
    (re.compile(r"\bmöcüzə\b|\bmocuze\b",     re.I), "Möcüzə"),
    (re.compile(r"\bzeytun\s*ba[ğg]lar[ıi]\b", re.I), "Zeytun Bağları"),
    (re.compile(r"\baz[əe]r\s*[şs][əe]k[əe]r\b|\bazerseker\b", re.I), "Azərşəkər"),
    (re.compile(r"\bkirlang[ıi][çc]\b|\bk[ıi]rlang[ıi][çc]\b|\bkırlanğıc\b", re.I), "Kırlangıç"),
    (re.compile(r"\bprostokvaşino\b|\bprostokvashino\b", re.I), "Prostokvaşino"),
    (re.compile(r"\bzolotoe\s*soln[ıi][şs]ko\b", re.I), "Zolotoe Solnışko"),
    (re.compile(r"\bkrestyanskoe\b",           re.I), "Krestyanskoe"),
    (re.compile(r"\bsavuşkin\b|\bsavushkin\b", re.I), "Savuşkin"),
    (re.compile(r"\bdoyaru[şs]ka\b",           re.I), "Doyaruşka"),
    (re.compile(r"\baz[əe]rs[üu]d\b",          re.I), "Azərsüd"),
    (re.compile(r"\baz[əe]rduz\b",             re.I), "Azərduz"),
    (re.compile(r"\bp[iı]nar\b",               re.I), "Pınar"),
    (re.compile(r"\băçim\b|\biçim\b",          re.I), "İçim"),
]


_FIRST_TOK = re.compile(
    r"^([A-Za-z\u018f\u0259\u0130\u0131\u00d6\u00f6\u00dc\u00fc"
    r"\u011e\u011f\u015e\u015f\u00c7\u00e7\u00d1\u00f1\u00c2\u00e2]+)"
)

def extract_brand(name_raw: object) -> tuple[str | None, str]:
    """
    Returns (brand, source) where source ∈ {'alias', 'list', 'first_token', 'unknown'}
    """
    if not isinstance(name_raw, str) or not name_raw.strip():
        return None, "unknown"

    raw = name_raw.strip()

    for rx, canon in _ALIASES:
        if rx.search(raw):
            return canon, "alias"

    folded = fold(raw)
    m = _BRAND_PAT.search(folded)
    if m:
        matched_fold = m.group(1).lower()
        canon = fold_to_canon.get(matched_fold)
        if canon:
            return canon, "list"

    m = _FIRST_TOK.match(raw.strip())
    if m:
        return m.group(1), "first_token"

    return None, "unknown"


df[["brand", "brand_source"]] = (
    df["product_name_raw"]
    .apply(lambda x: pd.Series(extract_brand(x)))
)


total    = len(df)
src_cnt  = df["brand_source"].value_counts()
nan_cnt  = df["brand"].isna().sum()

print(f"\n{'='*50}")
print("  BRAND EXTRACTION REPORT")
print(f"{'='*50}")
print(f"  Cemi mehsul    : {total}")
print(f"  Brand tapilan  : {total - nan_cnt}  ({(total-nan_cnt)/total*100:.1f}%)")
print(f"  Brand tapilmayan: {nan_cnt}")
print(f"\n  Mənbəyə görə:")
for src, cnt in src_cnt.items():
    print(f"    {src:15s}: {cnt}")
print(f"{'='*50}")


top = df["brand"].value_counts().head(20)
print(f"\nTop 20 brand:")
print(top.to_string())

ft = df[df["brand_source"] == "first_token"][["product_name_raw", "brand"]].head(30)
if not ft.empty:
    print(f"\nFirst-token (yoxla): {len(df[df['brand_source']=='first_token'])} məhsul — ilk 30:")
    print(ft.to_string(index=False))




  BRAND EXTRACTION REPORT
  Cemi mehsul    : 441
  Brand tapilan  : 441  (100.0%)
  Brand tapilmayan: 0

  Mənbəyə görə:
    list           : 336
    alias          : 77
    first_token    : 28

Top 20 brand:
brand
Milla               23
Azərsüd             13
Ankara              12
Final               11
Azərşəkər           11
Ariel               11
Femilea              9
Doymak               9
Persil               9
Atena                8
Oluna                8
Vanish               8
Perwoll              8
Shaan                7
Favelli              7
Barilla              7
Cleverest            7
Zolotoe Solnışko     6
Harika               6
Dr.Oetker            6

First-token (yoxla): 28 məhsul — ilk 30:
                                              product_name_raw          brand
                              Sahplov Basmati Düyü Banka 910qr        Sahplov
                            Sulan Qəhvəyi Basmati Düyü 1kq Pet          Sulan
                                           Sütaş

In [9]:
df = df[~df.astype(str).apply(lambda x: x.str.contains('Qabartma', case=False)).any(axis=1)]


In [10]:
df["brand"].isnull().sum()

np.int64(0)

In [ ]:

import re
import pandas as pd

SUB_RULES: list[tuple[str, re.Pattern]] = [

    ("Yuyucu toz", re.compile(
        r"\b(yuyucu\s*toz|toz\s*deterjan|deterjan"
        r"|persil|tide|era|bingo|sizlik|synergetic)\b",
        re.I
    )),
    ("Yuyucu maye", re.compile(
        r"\b(yuyucu\s*maye|qabyuyan\s*maye|qabyuyan|qab\s*yuyan"
        r"|perwoll|vanish|vernel|domestos|ace|ariel|calgon|cilit"
        r"|mr\s*muscle|bloom|crystalex|fairy"
        r"|yuyucu\s*gel|yuma|waschk[oö]nig|der\s*waschkonig|color\s*gel|universal\s*gel)\b",
        re.I
    )),

    ("Şəkər", re.compile(
        r"\b(şəkər|seker|qənd|qend|kəllə\s*qənd|kelle\s*qend"
        r"|nabat|noğul|nogul|draje|azərşəkər|azerseker)\b",
        re.I
    )),
    ("Duz", re.compile(
        r"\b(duz|xörək\s*duzu|xorek\s*duzu|süfrə\s*duzu|sufre\s*duzu"
        r"|limon\s*duzu|dəniz\s*duzu|deniz\s*duzu|himalay\s*duzu"
        r"|turşu\s*duzu|tursu\s*duzu|yodlu|azərduz|svan\s*duzu"
        r"|mikroelement|iri\s*duz|edviyat"
        r"|qaya\s*duzu|kristal\s*kaya|kaya\s*duzu|çankırı|pendir\s*turşusu"
        r"|deniz\s*duzu|duz\s*şüşə)\b",
        re.I
    )),

    ("Un", re.compile(
        r"\b(un|unu|buğda\s*unu|bugda\s*unu|misir\s*unu|qarğıdalı\s*unu"
        r"|qargidali\s*unu|düyü\s*unu|duyu\s*unu|kəpək\s*unu|nişasta"
        r"|nishasta|blincik|maya|pakmaya|saf\s*anlik"
        r"|qlutensiz|glutensiz|corek|cörək|rustico|multiqranul|salinis"
        r"|krendel|borodinskiy|deniz\s*yosunu)\b",
        re.I
    )),
    ("Düyü", re.compile(
        r"\b(düyü|duyu|basmati|basmat[iı]|sella"
        r"|qarabasıq|qarabasag|astara|hamtaz|səhriyyə"
        r"|bulg[əe]r|bulgur|yumru\s*düyü|has[ıi]mi|pəhriz\s*düyü"
        r"|səhriyə.*düyü|şamtaz|uzun\s*düyü)\b",
        re.I
    )),
    ("Makaron", re.compile(
        r"\b(makaron|spagetti|spaqetti|əriştə|eriste|vermişel|vermisel"
        r"|vermicelli|penne|fusilli|tortiglioni|rigate|bucatini|gomiti"
        r"|spaghetti|spaghettini|spaghettoni|bavette|cappelini|chifferi"
        r"|maccero|lasagne|lasangne"
        r"|tel\s*şehriye|sehriye|şehriyə|mutlu\s*ulduz"
        r"|fiyonk|burgu|qələm|qelem|boru|yuva|mant[iı]"
        r"|rollton|favelli|pasta\s*reggia|linguine"
        r"|farfalle|tagliatelle|lasagna)\b",
        re.I
    )),

    ("Ayran", re.compile(
        r"\b(ayran|dovga)\b",
        re.I
    )),
    ("Qatıq", re.compile(
        r"\b(qatıq|qatiq|qat[iı][gğ][iı]|yoqurt|yogurt|kefir"
        r"|ryajenka|r[eə]jenka|camış\s*qat|kənd\s*qat|fitnes\s*qat"
        r"|pudinq|kakao)\b",
        re.I
    )),
    ("Süd", re.compile(
        r"\b(süd|sud|milk|xama|qaymaq|qatılaşdırılmış\s*süd"
        r"|uht\s*süd|uht|südlü|südün|kokos\s*südü"
        r"|az[əe]rs[üu]d\s*\d|\d+\.?\d*%)\b",
        re.I
    )),

    ("Kərə yağı", re.compile(
        r"\b(kərə\s*yağı|kərəyağı|kere\s*yagi|eridilmiş\s*yağ"
        r"|əridilmiş\s*yağ|butter|margarin|marqarin"
        r"|mətbəx\s*yağ|xəmir\s*yağ|kərəli\s*yağ|fin\s*yağ"
        r"|82[\.,]5%|82%|79%"
        r"|anchor\s*yağ|president\s*yağ"
        r"|küncüt\s*yağ|üzumt[ou]mu?\s*yağ|duzsuz\s*yağ"
        r"|qat\s*qat\s*xəmir"
        r"|ağ\s*xəmir\s*yağı|xəmir\s*ücün\s*yağ)\b",
        re.I
    )),
    ("Zeytun yağı", re.compile(
        r"\b(zeytun\s*ya[gğ][ıi]|olive|pomace|sızma|zeytunyagi"
        r"|virgin|extra\s*virgin"
        r"|zeytun\s*yağ\b|yağlda\s*zeytun|yağ\s*500ml"
        r"|zira\s*olives|zeytun bağları|rafine\s*edilmiş"
        r"|qara\s*zeytun|tumlu\s*qara|olives\s*zeytun)\b",
        re.I
    )),
    ("Günəbaxan yağı", re.compile(
        r"\b(g[üu]n[əe]baxan\s*ya[gğ][ıi]|sunflower|gunebaxan)\b",
        re.I
    )),
    ("Qarğıdalı yağı", re.compile(
        r"\b(qar[ğıg]dal[ıi]\s*ya[gğ][ıi]|qar[ğg][iı]dal[iı]\s*ya[gğ][iı]"
        r"|qarğıdalı|qargidali|misir\s*ya[gğ][ıi]|corn\s*oil)\b",
        re.I
    )),
]


GROUP_MAP: dict[str, str] = {
    "Yuyucu toz":    "Non food",
    "Yuyucu maye":   "Non food",

    "Şəkər":         "Everyday essentials",
    "Duz":           "Everyday essentials",

    "Un":            "Core staples",
    "Düyü":          "Core staples",
    "Makaron":       "Core staples",

    "Ayran":         "Dairy",
    "Qatıq":         "Dairy",
    "Süd":           "Dairy",

    "Kərə yağı":     "Cooking fats",
    "Zeytun yağı":   "Cooking fats",
    "Günəbaxan yağı":"Cooking fats",
    "Qarğıdalı yağı":"Cooking fats",
}


df["category_sub"] = "Others"

for sub_label, rx in SUB_RULES:
    mask = df["name_lc"].apply(lambda s, _rx=rx: bool(_rx.search(s)))
    df.loc[mask & (df["category_sub"] == "Others"), "category_sub"] = sub_label

df["category_group"] = df["category_sub"].map(GROUP_MAP).fillna("Others")


total      = len(df)
others_cnt = (df["category_sub"] == "Others").sum()
ok_cnt     = total - others_cnt

print(f"\n{'='*50}")
print("  CATEGORY EXTRACTION REPORT")
print(f"{'='*50}")
print(f"  Cemi mehsul    : {total}")
print(f"  Kategoriyali   : {ok_cnt}  ({ok_cnt/total*100:.1f}%)")
print(f"  Others         : {others_cnt}")
print(f"{'='*50}")

print("\ncategory_sub dagilimi:")
print(df["category_sub"].value_counts().to_string())

print("\ncategory_group dagilimi:")
print(df["category_group"].value_counts().to_string())

if others_cnt:
    print(f"\nOthers-da qalan {others_cnt} mehsul:")
    print(
        df.loc[df["category_sub"] == "Others", "product_name_raw"]
        .to_string(index=False)
    )




  CATEGORY EXTRACTION REPORT
  Cemi mehsul    : 436
  Kategoriyali   : 410  (94.0%)
  Others         : 26

category_sub dagilimi:
category_sub
Makaron           47
Süd               45
Kərə yağı         42
Düyü              36
Yuyucu maye       34
Un                32
Zeytun yağı       30
Others            26
Qatıq             25
Yuyucu toz        25
Şəkər             22
Ayran             19
Günəbaxan yağı    19
Duz               17
Qarğıdalı yağı    17

category_group dagilimi:
category_group
Core staples           115
Cooking fats           108
Dairy                   89
Non food                59
Everyday essentials     39
Others                  26

Others-da qalan 26 mehsul:
                           Sab Gəncə Dovğası 200ml
                               Sab Dovğa Xan 200ml
                             Uzun Ömür Dovğa 200qr
                    Uzun Ömür İvanovka Dovğa 800qr
                             Final Mətbəx Yağı 2kq
                          Valio Fin Yağı 79% 200qr
    

In [12]:
df = df[~df.astype(str).apply(lambda x: x.str.contains('Küncüt', case=False)).any(axis=1)]
df = df[~df.astype(str).apply(lambda x: x.str.contains('Üzümtumu', case=False)).any(axis=1)]
df = df[~df.astype(str).apply(lambda x: x.str.contains('Dovğa', case=False)).any(axis=1)]

In [26]:
others_df = df[df["category_sub"] == "Others"]
display(others_df)

,scrape_run_id,scrape_ts_utc,snapshot_date,market,category_raw,product_name_raw,barcode,price_current_azn,price_old_azn,price_current_raw,...,unit_price_1kg,unit_price_1l,unit_price_base,unit_price_base_label,unit_price_raw,name_lc,brand,brand_source,category_sub,category_group


In [25]:
df.loc[[197,199,200,201], "category_sub"] = "makaron"
df.loc[[197,199,200,201], "category_group"] = "core staples"

In [23]:
indicates_to_drop=[283,288]
df=df.drop(indicates_to_drop)

In [21]:
df.loc[[320,332,333], "category_sub"] = "sud"
df.loc[[320,332,333], "category_group"] = "dairy"

In [19]:
df.loc[[433,434], "category_sub"] = "yuyucu toz"
df.loc[[433,434], "category_group"] = "non food"

In [17]:
df.loc[[453,457,460], "category_sub"] = "zeytunyagi"
df.loc[[453,457,460], "category_group"] = "cooking_fats"

In [ ]:
df.loc[[147,148,149,150,151,152,153,154], "category_sub"] = "kereyagi"
df.loc[[147,148,149,150,151,152,153,154], "category_group"] = "cooking_fats"

In [ ]:

import re
import pandas as pd

_SCI_RE = re.compile(r"^\d+\.?\d*[eE][+\-]?\d+$")


def clean_barcode(val: object) -> str | None:
    
    if pd.isna(val):
        return None

    s = str(val).strip()

    
    if _SCI_RE.match(s):
        try:
            s = str(int(float(s)))
        except (ValueError, OverflowError):
            pass  


    if len(s) == 14 and s.isdigit() and s.endswith("0"):
        s = s[:-1]

    
    s = s.replace(" ", "").strip()

    return s if s else None

df["barcode_clean"] = df["barcode"].apply(clean_barcode)


def _ean13_checkdigit_ok(s: str) -> bool:
    digits = [int(c) for c in s]
    total = sum(d * (3 if i % 2 else 1) for i, d in enumerate(digits[:-1]))
    check = (10 - (total % 10)) % 10
    return check == digits[-1]

def _ean8_checkdigit_ok(s: str) -> bool:
    digits = [int(c) for c in s]
    total = sum(d * (3 if i % 2 else 1) for i, d in enumerate(digits[:-1]))
    check = (10 - (total % 10)) % 10
    return check == digits[-1]


def is_valid_barcode(bc: object) -> bool:
    """
    True  → barcode etibarlıdır
    False → etibarsız (NaN, hərfli, yanlış uzunluq, check-digit xəta)
    """
    if bc is None or not isinstance(bc, str):
        return False

    s = bc.strip()

    if not s.isdigit():
        return False

    n = len(s)

    if n == 13:
        return _ean13_checkdigit_ok(s)
    elif n == 8:
        return _ean8_checkdigit_ok(s)
    elif n == 12:
        return _ean13_checkdigit_ok("0" + s)
    else:
        return n == 14 and s.isdigit()

df["valid_barcode"]      = df["barcode_clean"].apply(is_valid_barcode)
df["dq_invalid_barcode"] = (~df["valid_barcode"]).astype(int)

total        = len(df)
nan_cnt      = df["barcode_clean"].isna().sum()
valid_cnt    = df["valid_barcode"].sum()
invalid_cnt  = df["dq_invalid_barcode"].sum()

print(f"\n{'='*48}")
print("  BARCODE REPORT")
print(f"{'='*48}")
print(f"  Cemi sətir          : {total}")
print(f"  barcode_clean NaN   : {nan_cnt}")
print(f"  valid_barcode = True: {valid_cnt}  ({valid_cnt/total*100:.1f}%)")
print(f"  dq_invalid_barcode  : {invalid_cnt}")
print(f"{'='*48}")

invalid_df = df.loc[df["dq_invalid_barcode"] == 1,
                    ["product_name_raw", "barcode", "barcode_clean"]]
if not invalid_df.empty:
    print(f"\nEtibarsız {len(invalid_df)} barcode:")
    print(invalid_df.to_string(index=False))

len_counts = (
    df["barcode_clean"]
    .dropna()
    .apply(len)
    .value_counts()
    .sort_index()
)
print(f"\nBarcode uzunluq dağılımı:")
print(len_counts.to_string())




  BARCODE REPORT
  Cemi sətir          : 427
  barcode_clean NaN   : 8
  valid_barcode = True: 0  (0.0%)
  dq_invalid_barcode  : 427

Etibarsız 427 barcode:
                                              product_name_raw      barcode   barcode_clean
                                    Atena Ayran Nanəli 1.4% 1l 4.760088e+12 4760087900032.0
                                           Atena Ayran 1.4% 1l 4.760088e+12 4760087900148.0
                                      Milla Ayran Nanəli 200ml 4.760082e+12 4760081601799.0
                                             Milla Ayran 200ml 4.760082e+12 4760081601171.0
                                             Milla Ayran 375ml 4.760082e+12 4760081600747.0
                                      Milla Ayran Nanəli 375ml 4.760082e+12 4760081601805.0
                                       Milla Ayran Nanə ilə 1l 4.760082e+12 4760081601812.0
                                                Milla Ayran 1l 4.760082e+12 4760081600754.0
              

In [ ]:


import re, hashlib, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

_SCI = re.compile(r"^\d+\.?\d*[eE][+\-]?\d+$")

def clean_barcode(val):
    if pd.isna(val): return None
    s = str(val).strip()
    if _SCI.match(s):
        try: s = str(int(float(s)))
        except: pass
    if len(s) == 14 and s.isdigit() and s.endswith("0"):
        s = s[:-1]
    return s.replace(" ", "").strip() or None

df["barcode_clean"] = df["barcode"].apply(clean_barcode)

_UNITS  = r"(?:kg|kq|qram|qr|gr|g|litr|lt|ml|l|q)(?!\w)"
_MPFWD  = re.compile(rf"(\d+)\s*[x×*]\s*(\d+(?:[.,]\d+)?)\s*({_UNITS})", re.I)
_MPREV  = re.compile(rf"(\d+(?:[.,]\d+)?)\s*({_UNITS})\s*[x×*]\s*(\d+)", re.I)
_SINGLE = re.compile(rf"(\d+(?:[.,]\d+)?)\s*({_UNITS})", re.I)
_THOU   = re.compile(r"(\d)[.,](\d{3})(?!\d)")

def _parse(s):
    s = s.strip()
    if re.match(r"^\d+[.,]\d{3}$", s):
        return float(s.replace(".", "").replace(",", ""))
    return float(s.replace(",", "."))

def _base(qty, unit):
    u = unit.lower().rstrip()
    if u in ("kg","kq"):                    return qty*1000, "g"
    if u in ("g","gr","qr","qram","q"):     return qty,      "g"
    if u in ("l","lt","litr"):              return qty*1000, "ml"
    if u == "ml":                           return qty,      "ml"
    return np.nan, None

def _extract(text):
    if not isinstance(text, str) or not text.strip(): return np.nan, None
    t = text.lower().replace("\xa0"," ").replace(",",".")
    t = _THOU.sub(r"\1\2", t)
    m = _MPFWD.search(t)
    if m:
        bq, bu = _base(_parse(m.group(2)) * int(m.group(1)), m.group(3))
        if bu: return bq, bu
    m = _MPREV.search(t)
    if m:
        bq, bu = _base(_parse(m.group(1)) * int(m.group(3)), m.group(2))
        if bu: return bq, bu
    all_m = _SINGLE.findall(t)
    if all_m:
        bq, bu = _base(_parse(all_m[-1][0]), all_m[-1][1])
        if bu: return bq, bu
    return np.nan, None

df[["_qty","_unit"]] = df["product_name_raw"].apply(lambda x: pd.Series(_extract(x)))
df["std_qty"] = df.apply(
    lambda r: f"{int(r['_qty'])}{r['_unit']}"
              if pd.notna(r["_qty"]) and r["_unit"] else None, axis=1
)

_AZ = str.maketrans({
    "ə":"e","Ə":"e","ş":"s","Ş":"s","ı":"i","İ":"i",
    "ğ":"g","Ğ":"g","ç":"c","Ç":"c","ö":"o","Ö":"o","ü":"u","Ü":"u",
})

def _fold(s):
    if not isinstance(s, str): return ""
    return re.sub(r"\s+", " ", s.lower().translate(_AZ).strip())

def product_name_clean(s):
    if not isinstance(s, str) or not s.strip(): return None
    s = re.sub(r"[^\w\s.\-()]", "", s, flags=re.UNICODE)
    return re.sub(r"\s+", " ", s.strip()) or None

def name_norm(s):
    if not isinstance(s, str) or not s.strip(): return None
    return _fold(s) or None

def name_norm2(s):
    n = name_norm(s)
    if not n: return None
    return re.sub(r"\d+[.,]?\d*\s*(%|g|ml|l|kg|kq|qr|gr|litr|lt)?", "", n).strip() or None

df["product_name_clean"] = df["product_name_raw"].apply(product_name_clean)
df["name_norm"]          = df["product_name_raw"].apply(name_norm)
df["name_norm2"]         = df["product_name_raw"].apply(name_norm2)

df["product_canonical"] = df["category_raw"].apply(
    lambda x: _fold(str(x)) if pd.notna(x) else None
)

def md5_12(s): return hashlib.md5(str(s).encode("utf-8")).hexdigest()[:12]

df["identity_string"] = df.apply(
    lambda r: f"{r['name_norm'] or ''}|{r['category_raw'] or ''}", axis=1)
df["identity_hash"] = df["identity_string"].apply(md5_12)

df["identity_string_strong"] = df.apply(
    lambda r: f"{r['name_norm'] or ''}|{r['category_raw'] or ''}|{r['std_qty'] or ''}",
    axis=1)
df["identity_hash_strong"] = df["identity_string_strong"].apply(md5_12)

df["global_product_id"]     = df["barcode_clean"]
df["dq_missing_gpid"]       = df["global_product_id"].isna().astype(int)
df["gpid_hash"]             = df["global_product_id"].apply(
    lambda x: md5_12(x) if pd.notna(x) else None)
df["global_product_id_sku"] = df.apply(
    lambda r: f"{r['global_product_id']}|{r['std_qty']}"
              if pd.notna(r["global_product_id"]) else None, axis=1)

df["sku_str"] = df.apply(
    lambda r: (f"{r.get('scrape_run_id','?')}|{r['barcode_clean']}"
               if pd.notna(r.get("barcode_clean"))
               else f"{r.get('scrape_run_id','?')}|{r['identity_hash_strong']}"),
    axis=1)

def _sku_key(r):
    return (r["global_product_id"]    if pd.notna(r["global_product_id"]) else
            r["identity_hash_strong"] if pd.notna(r["identity_hash_strong"]) else
            r["barcode_clean"]        if pd.notna(r["barcode_clean"]) else
            r["sku_str"])

df["sku_key"] = df.apply(_sku_key, axis=1)


OUT_COLS = [
    "product_name_raw",
    "product_name_clean",
    "name_norm",
    "name_norm2",
    "std_qty",
    "product_canonical",
    "sku_key",
    "sku_str",
    "global_product_id",
    "global_product_id_sku",
    "gpid_hash",
    "identity_string",
    "identity_hash",
    "identity_string_strong",
    "identity_hash_strong",
    "dq_missing_gpid",
]

df_out = df[[c for c in OUT_COLS if c in df.columns]]


total = len(df_out)
print(f"\n{'='*50}")
print("  IDENTITY  REPORT")
print(f"{'='*50}")
print(f"  Cemi sətir           : {total}")
print(f"  global_product_id    : {df_out['global_product_id'].notna().sum()} ({df_out['global_product_id'].notna().sum()/total*100:.1f}%)")
print(f"  dq_missing_gpid      : {df_out['dq_missing_gpid'].sum()}")
print(f"  sku_key unikal       : {df_out['sku_key'].nunique()}")
print(f"  std_qty dolu         : {df_out['std_qty'].notna().sum()}")
print(f"{'='*50}")
print("\nNümunə (2 sətir):")
print(df_out[["product_name_raw","sku_key","identity_hash_strong","std_qty"]].head(2).to_string(index=False))




  IDENTITY  REPORT
  Cemi sətir           : 427
  global_product_id    : 419 (98.1%)
  dq_missing_gpid      : 8
  sku_key unikal       : 427
  std_qty dolu         : 427

Nümunə (2 sətir):
          product_name_raw         sku_key identity_hash_strong std_qty
Atena Ayran Nanəli 1.4% 1l 4760087900032.0         8d9dcbc5de71  1000ml
       Atena Ayran 1.4% 1l 4760087900148.0         bb67d75595ce  1000ml


In [ ]:
import pandas as pd
import numpy as np

print("======================================")
print("FINAL DATA QUALITY AUDIT STARTED")
print("======================================")

TOTAL_ROWS = len(df)

critical_cols = [
    "product_name_clean",
    "brand",
    "std_qty_total",
    "std_unit",
    "category_sub",
    "category_group",
    "global_product_id",
    "identity_hash_strong"
]

null_report = {}

for col in critical_cols:
    if col in df.columns:
        null_count = df[col].isna().sum()
        null_report[col] = null_count

print("\n--- Critical Null Report ---")
for k,v in null_report.items():
    print(f"{k}: {v} null ({round(v/TOTAL_ROWS*100,2)}%)")

FINAL DATA QUALITY AUDIT STARTED

--- Critical Null Report ---
product_name_clean: 0 null (0.0%)
brand: 0 null (0.0%)
std_qty_total: 3 null (0.7%)
std_unit: 3 null (0.7%)
category_sub: 0 null (0.0%)
category_group: 0 null (0.0%)
global_product_id: 8 null (1.87%)
identity_hash_strong: 0 null (0.0%)


In [ ]:

import re
import pandas as pd

def _ean13_ok(s: str) -> bool:
    d = [int(c) for c in s]
    t = sum(x * (3 if i % 2 else 1) for i, x in enumerate(d[:-1]))
    return (10 - t % 10) % 10 == d[-1]

def _ean8_ok(s: str) -> bool:
    d = [int(c) for c in s]
    t = sum(x * (3 if i % 2 else 1) for i, x in enumerate(d[:-1]))
    return (10 - t % 10) % 10 == d[-1]

def is_valid_ean(s: str) -> bool:
    if not s or not s.isdigit():
        return False
    n = len(s)
    if n == 13: return _ean13_ok(s)
    if n ==  8: return _ean8_ok(s)
    if n == 12: return _ean13_ok("0" + s)
    return False

_TAIL_RE = re.compile(r"(?<!\d)(\d{8}|\d{12}|\d{13})(?!\d)\s*$")

def extract_embedded_barcode(name: str):
    """
    Məhsul adının sonundakı etibarlı EAN rəqəmini çıxarır.
    Tapılmırsa None qaytarır.
    """
    if not isinstance(name, str):
        return None
    m = _TAIL_RE.search(name.strip())
    if m:
        candidate = m.group(1)
        if is_valid_ean(candidate):
            return candidate
    return None



for col in ("global_product_id", "identity_hash_strong", "dq_missing_gpid"):
    if col not in df.columns:
        df[col] = None

mask_nan = df["global_product_id"].isna()
print(f"[GPID] NaN olan sətir sayı: {mask_nan.sum()}")

df.loc[mask_nan, "_embedded_bc"] = (
    df.loc[mask_nan, "product_name_raw"].apply(extract_embedded_barcode)
)

found_embedded = mask_nan & df["_embedded_bc"].notna()
df.loc[found_embedded, "global_product_id"] = df.loc[found_embedded, "_embedded_bc"]

print(f"\n[ADDIM 1] Ada gömülü barcode tapılan: {found_embedded.sum()}")
if found_embedded.sum():
    print(df.loc[found_embedded, ["product_name_raw", "global_product_id"]].to_string(index=True))

df.drop(columns=["_embedded_bc"], inplace=True)

mask_still_nan = df["global_product_id"].isna()

if "identity_hash_strong" not in df.columns:
    import hashlib
    def md5_12(s): return hashlib.md5(str(s).encode()).hexdigest()[:12]
    df["identity_hash_strong"] = df.apply(
        lambda r: md5_12(f"{r.get('name_norm','')}|{r.get('category_raw','')}|{r.get('std_qty','')}"),
        axis=1
    )

df.loc[mask_still_nan, "global_product_id"] = (
    "IHS_" + df.loc[mask_still_nan, "identity_hash_strong"].fillna("unknown")
)
print(f"\n[ADDIM 2] identity_hash_strong ilə dolduruldu: {mask_still_nan.sum()}")
if mask_still_nan.sum():
    print(df.loc[mask_still_nan, ["product_name_raw", "global_product_id"]].to_string(index=True))

df["dq_missing_gpid"] = df["global_product_id"].isna().astype(int)

def _sku_key(r):
    gpid = r.get("global_product_id")
    if pd.notna(gpid) and not str(gpid).startswith("IHS_"):
        return str(gpid)                     
    ihs  = r.get("identity_hash_strong")
    if pd.notna(ihs):
        return str(ihs)
    bc   = r.get("barcode_clean")
    if pd.notna(bc):
        return str(bc)
    return r.get("sku_str", "unknown")

if "sku_key" in df.columns:
    mask_was_nan = mask_nan                 
    df.loc[mask_was_nan, "sku_key"] = df.loc[mask_was_nan].apply(_sku_key, axis=1)

# YEKUN 
print(f"\n{'='*52}")
print("  GPID FIX  REPORT")
print(f"{'='*52}")
print(f"  Cemi sətir              : {len(df)}")
print(f"  global_product_id dolu  : {df['global_product_id'].notna().sum()}")
print(f"  dq_missing_gpid=1       : {df['dq_missing_gpid'].astype(int).sum()}")
print(f"  Ada gömülü barcode      : {found_embedded.sum()}")
print(f"  IHS_ fallback           : {mask_still_nan.sum()}")
print(f"{'='*52}")



[GPID] NaN olan sətir sayı: 8

[ADDIM 1] Ada gömülü barcode tapılan: 0

[ADDIM 2] identity_hash_strong ilə dolduruldu: 8
                                   product_name_raw global_product_id
29                             Dr Milk Ayran 900 ml  IHS_fb40f2cf16f7
101            Divo 1lt Günəbaxan Yağı (Şüşə Qabda)  IHS_7add8b4bd6c4
135                         Gerda Kərə Yağı 1kq 118  IHS_2843aa83b3c6
297  Danone Disney Çiyələkli Uht Süd 180qr 86902618  IHS_53221977077e
460                 Oli̇tali̇a Təmiz Zeytun Yağl 1L  IHS_5309b9bcceb1
461            Olive Oil Extra Virgin 500ml Sh 0022  IHS_a2551f8bf6ba
462            Olive Oil Extra Virgin 250ml Sh 0039  IHS_6259021291a3
463           Olitalia Pomace Zeytun Yağı Şüşə 1 Lt  IHS_8688cf306c14

  GPID FIX  REPORT
  Cemi sətir              : 427
  global_product_id dolu  : 427
  dq_missing_gpid=1       : 0
  Ada gömülü barcode      : 0
  IHS_ fallback           : 8


In [31]:
df.loc[
    df["global_product_id"].isna(),
    ["product_name_raw", "category_raw"]
]

,product_name_raw,category_raw


In [ ]:

import re
import pandas as pd
import numpy as np

_Q_RE = re.compile(
    r"(\d+(?:[.,]\d{3})?)"   
    r"\s*q\b",                
    re.I
)

def _parse_q(text):
    """
    Yalnız q-unit-li miqdarı axtar.
    1.820q → 1820, 375q → 375, 380 q → 380
    """
    if not isinstance(text, str):
        return np.nan

    t = text.lower().replace("\xa0", " ")

    m = _Q_RE.search(t)
    if not m:
        return np.nan

    raw = m.group(1)                    
    raw_clean = raw.replace(".", "").replace(",", "")   
    try:
        return float(raw_clean)
    except ValueError:
        return np.nan


TARGETS = [
    "Astara Düyüsu Pəhriz Düyüsu 375q",
    "Düyü Tamasa Miad 910q",
    "Tamasa Miad Düyü 1.820q Pet",
    "Doyaruşka Kərə Yağı 380 q",
]

mask = df["product_name_raw"].isin(TARGETS)
print(f"\nHədəf sətirləri tapıldı: {mask.sum()} / {len(TARGETS)}")

print("\n--- Hər məhsul üçün nəticə ---")
for name in TARGETS:
    result = _parse_q(name)
    print(f"  {name!r:48s} → {result} g")

df.loc[mask, "std_qty_total"] = df.loc[mask, "product_name_raw"].apply(_parse_q)
df.loc[mask, "std_unit"]      = "g"
df.loc[mask, "std_qty"]       = df.loc[mask].apply(
    lambda r: f"{int(float(r['std_qty_total']))}g"
              if pd.notna(r.get("std_qty_total")) else None,
    axis=1
)
df.loc[mask, "dq_missing_unit"] = 0

SHOW_COLS = ["product_name_raw","category_raw","std_qty_total","std_unit","std_qty"]
show_cols = [c for c in SHOW_COLS if c in df.columns]

print("\n--- Düzəldilmiş sətirlərin dəyərləri ---")
print(df.loc[mask, show_cols].to_string(index=True))






Hədəf sətirləri tapıldı: 3 / 4

--- Hər məhsul üçün nəticə ---
  'Astara Düyüsu Pəhriz Düyüsu 375q'               → 375.0 g
  'Düyü Tamasa Miad 910q'                          → 910.0 g
  'Tamasa Miad Düyü 1.820q Pet'                    → 1820.0 g
  'Doyaruşka Kərə Yağı 380 q'                      → 380.0 g

--- Düzəldilmiş sətirlərin dəyərləri ---
                product_name_raw category_raw  std_qty_total std_unit std_qty
61         Düyü Tamasa Miad 910q         duyu          910.0        g    910g
62   Tamasa Miad Düyü 1.820q Pet         duyu         1820.0        g   1820g
116    Doyaruşka Kərə Yağı 380 q     kereyagi          380.0        g    380g


In [33]:
df.loc[
    df["std_unit"].isna(),
    ["product_name_raw", "category_raw"]
]

,product_name_raw,category_raw


In [ ]:
import pandas as pd
import numpy as np

print("======================================")
print("FINAL DATA QUALITY AUDIT STARTED")
print("======================================")

TOTAL_ROWS = len(df)


critical_cols = [
    "product_name_clean",
    "brand",
    "std_qty_total",
    "std_unit",
    "category_sub",
    "category_group",
    "global_product_id",
    "identity_hash_strong"
]

null_report = {}

for col in critical_cols:
    if col in df.columns:
        null_count = df[col].isna().sum()
        null_report[col] = null_count

print("\n--- Critical Null Report ---")
for k,v in null_report.items():
    print(f"{k}: {v} null ({round(v/TOTAL_ROWS*100,2)}%)")

FINAL DATA QUALITY AUDIT STARTED

--- Critical Null Report ---
product_name_clean: 0 null (0.0%)
brand: 0 null (0.0%)
std_qty_total: 0 null (0.0%)
std_unit: 0 null (0.0%)
category_sub: 0 null (0.0%)
category_group: 0 null (0.0%)
global_product_id: 0 null (0.0%)
identity_hash_strong: 0 null (0.0%)


In [35]:
import re

def normalize_text(s):
    if pd.isna(s):
        return ""
    
    s = str(s).lower()
    
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9 əöğçşı]", "", s)
    
    return s.strip()


if "product_name_raw" in df.columns:

    df["product_name_clean"] = df["product_name_raw"].astype(str).str.strip()

    
    df["name_norm"] = df["product_name_clean"].apply(normalize_text)

   
    df["product_canonical"] = df["name_norm"]
else:
    print("Xəta: 'product_name_raw' sütunu tapılmadı!")

In [ ]:
import pandas as pd
import numpy as np

print("======================================")
print("FINAL DATA QUALITY AUDIT STARTED")
print("======================================")

TOTAL_ROWS = len(df)


critical_cols = [
    "product_name_clean",
    "brand",
    "std_qty_total",
    "std_unit",
    "category_sub",
    "category_group",
    "global_product_id",
    "identity_hash_strong"
]

null_report = {}

for col in critical_cols:
    if col in df.columns:
        null_count = df[col].isna().sum()
        null_report[col] = null_count

print("\n--- Critical Null Report ---")
for k,v in null_report.items():
    print(f"{k}: {v} null ({round(v/TOTAL_ROWS*100,2)}%)")

FINAL DATA QUALITY AUDIT STARTED

--- Critical Null Report ---
product_name_clean: 0 null (0.0%)
brand: 0 null (0.0%)
std_qty_total: 0 null (0.0%)
std_unit: 0 null (0.0%)
category_sub: 0 null (0.0%)
category_group: 0 null (0.0%)
global_product_id: 0 null (0.0%)
identity_hash_strong: 0 null (0.0%)


In [ ]:

empty_report = {}

for col in critical_cols:
    if col in df.columns:
        empty_count = (df[col].astype("string").str.strip() == "").sum()
        empty_report[col] = empty_count

print("\n--- Empty String Report ---")
for k,v in empty_report.items():
    print(f"{k}: {v} empty")


--- Empty String Report ---
product_name_clean: 0 empty
brand: 0 empty
std_qty_total: 0 empty
std_unit: 0 empty
category_sub: 0 empty
category_group: 0 empty
global_product_id: 0 empty
identity_hash_strong: 0 empty


In [ ]:

dup_gpid = df["global_product_id"].duplicated().sum()

print("\n--- Duplicate GPID ---")
print(f"Duplicate GPID rows: {dup_gpid}")



--- Duplicate GPID ---
Duplicate GPID rows: 0


In [ ]:

barcode_conflict = df[
    (df["barcode_clean"].notna()) &
    (df["global_product_id"] != df["barcode_clean"])
]

print("\n--- Barcode Priority Conflict ---")
print(f"Conflicts: {len(barcode_conflict)}")


--- Barcode Priority Conflict ---
Conflicts: 0


In [40]:
import re

def normalize_text(s):
    if pd.isna(s):
        return ""
    
    s = str(s).lower()
    
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9 əöğçşı]", "", s)
    
    return s.strip()



if "product_name_raw" in df.columns:

    df["product_name_clean"] = df["product_name_raw"].astype(str).str.strip()

    
    df["name_norm"] = df["product_name_clean"].apply(normalize_text)

   
    df["product_canonical"] = df["name_norm"]
else:
    print("Xəta: 'product_name_raw' sütunu tapılmadı!")

In [ ]:
def get_col(df, col):
    if col in df.columns:
        return df[col]
    return pd.Series([pd.NA] * len(df), index=df.index, dtype="string")


def clean_barcode_series(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s = (
        s.str.strip()
         .str.replace(r"\.0$", "", regex=True)
         .str.replace(r"\D", "", regex=True)
    )
    s = s.replace(["", "None", "nan", "<NA>"], pd.NA)
    return s

def is_valid_ean(code) -> bool:
    if pd.isna(code):
        return False
    code = str(code)
    if len(code) not in (8, 12, 13, 14):
        return False
    if not code.isdigit():
        return False

    digits = [int(c) for c in code]
    check = digits[-1]
    body = digits[:-1]

    
    body_rev = body[::-1]
    total = 0
    for i, d in enumerate(body_rev, start=1):
        total += d * 3 if i % 2 == 1 else d 
    calc = (10 - (total % 10)) % 10
    return calc == check

In [ ]:

std_qty_str = (
    get_col(df, "std_qty_total")
    .astype("Float64")
    .astype("string")
    .replace(["<NA>","nan","None"], "")
)

brand = get_col(df, "brand").astype("string").fillna("")

df["identity_string_strong"] = (
    df["product_canonical"] + "|" +
    brand + "|" +
    std_qty_str + "|" +
    df["std_unit"].astype("string").fillna("")
)

df["identity_hash_strong"] = df["identity_string_strong"].apply(
    lambda x: hashlib.sha256(str(x).encode("utf-8")).hexdigest()
)

In [ ]:

collision_check = (
    df.groupby("identity_hash_strong")["product_name_clean"]
    .nunique()
)

identity_collisions = (collision_check > 1).sum()

print("\n--- Identity Collision ---")
print(f"Strong identity collisions: {identity_collisions}")


--- Identity Collision ---
Strong identity collisions: 0


In [ ]:

if "price_current_azn" in df.columns:
    negative_price = (df["price_current_azn"] < 0).sum()
else:
    negative_price = 0

print("\n--- Price Sanity ---")
print(f"Negative prices: {negative_price}")


--- Price Sanity ---
Negative prices: 0


In [ ]:

total_issues = (
    sum(null_report.values()) +
    sum(empty_report.values()) +
    dup_gpid +
    len(barcode_conflict) +
    identity_collisions +
    negative_price
)

print("\n======================================")
if total_issues == 0:
    print("DATA STATUS: ✅ CLEAN - SAFE TO SAVE")
else:
    print("DATA STATUS: ⚠ CHECK WARNINGS BEFORE SAVE")
print("======================================")




DATA STATUS: ✅ CLEAN - SAFE TO SAVE


In [46]:
save_path = "bravo_13fevaxsam_clean_final.csv"

df.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"\nSaved to: {save_path}")
print("Rows:", len(df))


Saved to: bravo_13fevaxsam_clean_final.csv
Rows: 427


In [ ]:
import os
from datetime import datetime

today = "2026_02_13"


silver_dir = "silver_layer"
os.makedirs(silver_dir, exist_ok=True)


save_path = f"{silver_dir}/araz_silver_{today}.csv"


df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Silver layer created:")
print(save_path)
print("Rows:", len(df))

Silver layer created:
silver_layer/araz_silver_2026_02_13.csv
Rows: 427
